In [1]:
import numpy as np
import cvxpy as cvx

In [12]:
import cvxpy as cp
import numpy as np

N = 20          # 예측 horizon
dt = 1.0        # time step
m = 300.0       # vehicle mass

v = cp.Variable(N + 1)
s = cp.Variable(N + 1)
E = cp.Variable(N + 1)
u = cp.Variable(N)
u_bar = np.ones(N)*200.0

v0 = 60/3.6
s0 = 0.0
E0 = 5_000_000.0

v_ref = np.ones(N + 1) * 60/3.6
GHI = np.ones(N) * 800.0
slope = np.zeros(N)

qv = 10.0
qE = 1e-8
ru = 1e-3

constraints = [
    v[0] == v0,
    s[0] == s0,
    E[0] == E0
]

cost = 0

for k in range(N):
    F_roll = 0.005 * m * 9.81
    F_slope = m * 9.81 * slope[k]

    # 단순화를 위해 공기저항은 기준속도 근처에서 상수처럼 처리
    # F_aero = 0.5 * 1.2 * 0.12 * 1.0 * (v_ref[k] ** 2)

    F_rest = F_roll + F_slope

    constraints += [
        v[k+1] == v[k]*(1-dt*0.5*1.2*0.12*v_ref[k]/m) + (dt/m)*u[k] - (dt/m)*F_rest,
        s[k+1] == s[k] + dt * v[k],
    ]

    # P_motor = u[k] * v_ref[k]
    #P_solar = 0.22 * 4.0 * GHI[k]

    constraints += [
        E[k+1] == E[k] - dt*(u_bar[k])*v[k] + -dt*(v_ref[k])*u[k] + dt*(0.22*4.0-1)
    ]

    constraints += [
        60/3.6 <= v[k],
        v[k] <= 100/3.6,
        0 <= u[k],
        u[k] <= 1000,
        E[k] >= 1_000_000
    ]

    cost += qv * cp.square(v[k] - v_ref[k])
    cost += qE * cp.square(E[k] - 3_000_000)
    cost += ru * cp.square(u[k])

problem = cp.Problem(cp.Minimize(cost), constraints)
problem.solve(solver=cp.OSQP)

print("optimal first control:", u.value[0])
print("predicted speed:", v.value)
print("predicted energy:", E.value)

optimal first control: 488.90382140907536
predicted speed: [16.45273961 17.74980641 19.02527308 20.28192188 21.52277827 22.75113354
 23.97053229 25.18472738 26.39760849 27.61311463 28.8351424  30.06746333
 31.31366214 32.57710526 33.86094493 35.16815892 36.50162108 37.86419305
 37.58039032 37.34009868 37.1416896 ]
predicted energy: [4999999.99728633 4988560.38960895 4976881.08877183 4964966.80257099
 4952820.77034566 4940444.44592163 4927837.32093052 4914996.91007903
 4901918.89975949 4888597.44021658 4875025.54197846 4861195.52225398
 4847099.43896486 4832729.45069308 4818078.05051906 4803138.1394795
 4787902.92874917 4772365.68514333 4764911.52958832 4757451.44436507
 4749983.29804727]


C:\Users\dlwns\AppData\Local\Temp\ipykernel_22960\33298302.py:68: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  problem.solve(solver=cp.OSQP)


In [10]:
np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])[:, 2]

array([ 3,  6,  9, 12])